In [ ]:
# imports
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from tqdm.notebook import tqdm

# load data
df = pd.read_csv('merged_data.csv', encoding='latin1')

# parse labels
def parseSpecificLabels(mediumString):
    mediumString = str(mediumString).upper()
    match = re.search(r'GLUCOSE (\d+)\s*MM', mediumString)
    if match:
        return f"Glucose_{match.group(1)}mM"
    if '100IPA' in medium_string: return 'IPA_100'
    if '50IPA50DI' in medium_string: return 'IPA_50'
    if '100ACETONE' in medium_string: return 'Acetone_100'
    if '50DI50ACETONE' in medium_string: return 'Acetone_50'
    if 'DI WATER' in medium_string or 'WATER' == medium_string: return 'DI_Water'
    if 'AIR' in medium_string: return 'Air'
    return 'Other'

df['specific_label'] = df['medium'].apply(parseSpecificLabels)
df.head()

In [ ]:
# sliding window
windowSize = 2048
step = 128
processedData = []

for _, group in tqdm(df.groupby('chunk_timestamp'), desc="Processing Sweeps"):
    label = group['specific_label'].iloc[0]
    signal = group['amplitude_v'].values
    
    for i in range(0, len(signal) - windowSize, step):
        window = signal[i : i + windowSize]
        rowData = {'specific_label': label}
        rowData.update({f'bin_{j}': val for j, val in enumerate(window)})
        processedData.append(rowData)

# samples per class
uprDataDf = pd.DataFrame(processedData)
print("Samples per class\n")
print(uprDataDf['specific_label'].value_counts())

In [ ]:
# residual block
def resnetBlock(x, filters, kernelSize=3, strides=1):
    shortcut = x
    x = layers.Conv1D(filters, kernelSize, strides=strides, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv1D(filters, kernelSize, padding='same')(x)
    x = layers.BatchNormalization()(x)
    if strides > 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv1D(filters, 1, strides=strides, padding='same')(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)
    return x

# build encoder
def buildUprEncoder():
    inputShape = (2048, 1)
    inputs = keras.Input(shape=inputShape)
    x = layers.Conv1D(64, 7, strides=2, padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(3, strides=2, padding='same')(x)
    x = resnetBlock(x, 64); x = resnetBlock(x, 64)
    x = resnetBlock(x, 128, strides=2); x = resnetBlock(x, 128)
    x = resnetBlock(x, 256, strides=2); x = resnetBlock(x, 256)
    x = layers.GlobalAveragePooling1D()(x)
    encoder = keras.Model(inputs, x, name='upr_encoder')
    return encoder

# create model
uprEncoder = buildUprEncoder()
uprEncoder.summary()

In [ ]:
# add noise
def addNoise(signal, noiseLevel=0.02):
    noise = tf.random.normal(shape=tf.shape(signal), stddev=noiseLevel)
    return signal + noise

# scale amplitude
def scaleAmplitude(signal, minScale=0.8, maxScale=1.2):
    scale = tf.random.uniform((), minScale, maxScale)
    return signal * scale

# spectral mask
def spectralMask(signal, maskRatio=0.05):
    signalLength = tf.shape(signal)[1]
    maskSize = tf.cast(tf.cast(signalLength, tf.float32) * maskRatio, tf.int32)
    maskStart = tf.random.uniform((), 0, signalLength - maskSize, dtype=tf.int32)
    maskShape = tf.stack([1, maskSize, 1])
    maskStartShape = tf.stack([1, maskStart, 1])
    remainingShape = tf.stack([1, signalLength - maskStart - maskSize, 1])
    mask = tf.concat([
        tf.ones(maskStartShape, dtype=tf.float32),
        tf.zeros(maskShape, dtype=tf.float32),
        tf.ones(remainingShape, dtype=tf.float32)
    ], axis=1)
    return signal * mask

# random augmentations
def augmentSignal(signal):
    signal = tf.cond(tf.random.uniform(()) > 0.5, lambda: addNoise(signal), lambda: signal)
    signal = tf.cond(tf.random.uniform(()) > 0.5, lambda: scaleAmplitude(signal), lambda: signal)
    signal = tf.cond(tf.random.uniform(()) > 0.5, lambda: spectralMask(signal), lambda: signal)
    return signal

In [ ]:
# projection head
projectionHead = keras.Sequential([
    keras.Input(shape=(256,)),
    layers.Dense(256, activation='relu'),
    layers.Dense(128),
], name='projection_head')

# nt-xent loss
def ntXentLoss(projectedV1, projectedV2, temperature=0.1):
    batchSize = tf.shape(projectedV1)[0]
    v1Normalized = tf.math.l2_normalize(projectedV1, axis=1)
    v2Normalized = tf.math.l2_normalize(projectedV2, axis=1)
    projections = tf.concat([v1Normalized, v2Normalized], axis=0)
    similarityMatrix = tf.matmul(projections, projections, transpose_b=True)
    labels = tf.range(batchSize)
    labels = tf.concat([labels, labels], axis=0)
    mask = tf.one_hot(labels, depth=batchSize * 2)
    positives = similarityMatrix[tf.cast(mask, dtype=tf.bool)]
    negatives = similarityMatrix[~tf.cast(mask, dtype=tf.bool)]
    negatives = tf.reshape(negatives, (batchSize * 2, batchSize * 2 - 1))
    logits = tf.concat([tf.expand_dims(positives, 1), negatives], axis=1)
    logits /= temperature
    return keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)

# simclr class
class SimCLR(keras.Model):
    def __init__(self, encoder, projector, augmentFunc):
        super().__init__()
        self.encoder = encoder
        self.projector = projector
        self.augmentFunc = augmentFunc

    def compile(self, optimizer, lossFn):
        super().compile()
        self.optimizer = optimizer
        self.lossFn = lossFn

    def train_step(self, data):
        view1 = self.augmentFunc(data)
        view2 = self.augmentFunc(data)
        with tf.GradientTape() as tape:
            h1 = self.encoder(view1, training=True)
            h2 = self.encoder(view2, training=True)
            z1 = self.projector(h1, training=True)
            z2 = self.projector(h2, training=True)
            loss = self.lossFn(z1, z2)
        trainableVars = self.encoder.trainable_variables + self.projector.trainable_variables
        gradients = tape.gradient(loss, trainableVars)
        self.optimizer.apply_gradients(zip(gradients, trainableVars))
        return {'loss': loss}

# init model
simclrModel = SimCLR(uprEncoder, projectionHead, augmentSignal)

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint, EarlyStopping

# extract features
features = uprDataDf.iloc[:, 1:].values.astype('float32')

# scale data
scaler = StandardScaler()
xTrainScaled = scaler.fit_transform(features)
xTrain = xTrainScaled.reshape((xTrainScaled.shape[0], xTrainScaled.shape[1], 1))

print(f"Total training samples: {len(xTrain)}")

# hyperparameters
BATCH_SIZE = 32
EPOCHS = 200
LEARNING_RATE = 0.001

reduceLr = ReduceLROnPlateau(
    monitor='loss',
    factor=0.2,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

modelCheckpoint = ModelCheckpoint(
    filepath='best_simclr_model.keras',
    monitor='loss',
    save_best_only=True,
    save_weights_only=False,
    verbose=1
)

earlyStopping = EarlyStopping(
    monitor='loss',
    patience=15,
    verbose=1,
    restore_best_weights=False
)

simclrModel.compile(
    optimizer=keras.optimizers.RMSprop(learning_rate=0.001, rho=0.9, epsilon=1e-7),
    lossFn=ntXentLoss
)

trainDataset = tf.data.Dataset.from_tensor_slices(xTrain).shuffle(len(xTrain)).batch(BATCH_SIZE, drop_remainder=True)

print("\nStarting training...")
history = simclrModel.fit(
    trainDataset,
    epochs=EPOCHS,
    callbacks=[reduceLr, modelCheckpoint, earlyStopping]
)

print("\nTraining complete.")

In [ ]:
# load encoder
inferenceEncoder = buildUprEncoder()
inferenceProjector = keras.Sequential([
    keras.Input(shape=(256,)),
    layers.Dense(256, activation='relu'),
    layers.Dense(128),
], name='projection_head_inference')

simclrModelForLoading = SimCLR(inferenceEncoder, inferenceProjector, augmentSignal)
simclrModelForLoading.load_weights('best_upr_encoder_weights.h5')
evalEncoder = simclrModelForLoading.encoder
evalEncoder.trainable = False

# generate embeddings
allSignalsRaw = uprDataDf.iloc[:, 1:].values.astype('float32')
allSignalsScaled = scaler.transform(allSignalsRaw)
allSignals = np.expand_dims(allSignalsScaled, axis=-1)
uprEmbeddings = evalEncoder.predict(allSignals)

# linear probe
evalDf = pd.DataFrame(uprEmbeddings)
evalDf['specific_label'] = uprDataDf['specific_label'].values
classCounts = evalDf['specific_label'].value_counts()
wellPopulatedClasses = classCounts[classCounts > 1].index
filteredEvalDf = evalDf[evalDf['specific_label'].isin(wellPopulatedClasses)]

X = filteredEvalDf.drop('specific_label', axis=1)
le = LabelEncoder()
y = le.fit_transform(filteredEvalDf['specific_label'])

xTrain, xTest, yTrain, yTest = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

linearModel = LogisticRegression(random_state=42, max_iter=2000)
linearModel.fit(xTrain, yTrain)
yPred = linearModel.predict(xTest)
accuracy = accuracy_score(yTest, yPred)

# results
report = classification_report(yTest, yPred, target_names=le.classes_)
print(f"--> Accuracy (Advanced UPR): {accuracy:.4f}")
print("\nDetailed Classification Report:\n")
print(report)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
import pandas as pd
import numpy as np

# t-sne setup
embeddings = uprEmbeddings
labels = le.classes_
numericLabels = y

print("Running t-SNE...")
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000, init='pca', learning_rate='auto')
embeddings2d = tsne.fit_transform(embeddings)

# plot dataframe
plotDf = pd.DataFrame({
    't-SNE Component 1': embeddings2d[:, 0],
    't-SNE Component 2': embeddings2d[:, 1],
    'Liquid': [labels[i] for i in numericLabels]
})

print("Generating cluster plot...")
plt.style.use('default')
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'stix'

fig, ax = plt.subplots(figsize=(10, 8))
palette = sns.color_palette("colorblind", n_colors=len(labels))

for i, liquid in enumerate(labels):
    subset = plotDf[plotDf['Liquid'] == liquid]
    ax.scatter(
        subset['t-SNE Component 1'],
        subset['t-SNE Component 2'],
        color=palette[i],
        label=liquid,
        alpha=0.85,
        s=80
    )

ax.set_title('t-SNE Visualization of PMUT Signal Embeddings', fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('t-SNE Component 1', fontsize=14)
ax.set_ylabel('t-SNE Component 2', fontsize=14)
ax.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

legend = ax.legend(title='Liquid Type', fontsize=11, title_fontsize=13, bbox_to_anchor=(1.05, 1), loc='upper left')
legend.get_frame().set_edgecolor('lightgray')

plt.savefig('tsne_cluster_visualization_publication.png', dpi=300, bbox_inches='tight')
plt.show()
print("\nPlot saved as 'tsne_cluster_visualization_publication.png'")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# plot style
plt.style.use('default')
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'stix'

# confusion matrix
cm = confusion_matrix(yTest, yPred, labels=le.transform(le.classes_))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)

fig, ax = plt.subplots(figsize=(10, 10))
disp.plot(
    ax=ax,
    cmap=plt.cm.Blues,
    xticks_rotation='vertical',
    values_format='d'
)

ax.set_title('Classifier Confusion Matrix', fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Predicted Label', fontsize=14)
ax.set_ylabel('True Label', fontsize=14)
ax.grid(False)

cbar = ax.figure.get_axes()[1]
cbar.set_ylabel('Number of Samples', fontsize=12)

plt.tight_layout()
plt.savefig('confusion_matrix_publication.png', dpi=300, bbox_inches='tight')
plt.show()
print("\nPlot saved as 'confusion_matrix_publication.png'")

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np
import pandas as pd
import re
from tqdm import tqdm

# load data
df = pd.read_csv('merged_data.csv', encoding='latin1')

# parse labels
def parseSpecificLabels(mediumString):
    mediumString = str(mediumString).upper()
    match = re.search(r'GLUCOSE (\d+)\s*MM', mediumString)
    if match:
        return f"Glucose_{match.group(1)}mM"
    if 'DI WATER' in mediumString or 'WATER' == mediumString: return 'DI_Water'
    if 'AIR' in mediumString: return 'Air'
    return 'Other'

df['specific_label'] = df['medium'].apply(parseSpecificLabels)

# sliding window
windowSize = 2048
step = 128
processedData = []

for _, group in tqdm(df.groupby('chunk_timestamp'), desc="Processing Sweeps"):
    label = group['specific_label'].iloc[0]
    signal = group['amplitude_v'].values
    for i in range(0, len(signal) - windowSize, step):
        window = signal[i : i + windowSize]
        rowData = {'specific_label': label}
        rowData.update({f'bin_{j}': val for j, val in enumerate(window)})
        processedData.append(rowData)

uprDataDf = pd.DataFrame(processedData)

# augmentation functions
def addNoise(signal, noiseLevel=0.02):
    noise = tf.random.normal(shape=tf.shape(signal), stddev=noiseLevel)
    return signal + noise

def scaleAmplitude(signal, minScale=0.8, maxScale=1.2):
    scale = tf.random.uniform((), minScale, maxScale)
    return signal * scale

def spectralMask(signal, maskRatio=0.05):
    if len(tf.shape(signal)) == 2:
        signal = signal[tf.newaxis, :, :]
    signalLength = tf.shape(signal)[1]
    maskSize = tf.cast(tf.cast(signalLength, tf.float32) * maskRatio, tf.int32)
    maskStart = tf.random.uniform((), 0, signalLength - maskSize, dtype=tf.int32)
    part1 = tf.ones([1, maskStart, 1], dtype=tf.float32)
    part2 = tf.zeros([1, maskSize, 1], dtype=tf.float32)
    part3 = tf.ones([1, signalLength - maskStart - maskSize, 1], dtype=tf.float32)
    mask = tf.concat([part1, part2, part3], axis=1)
    return signal * mask

# apply augmentations
originalSignalNp = uprDataDf.iloc[0, 1:].values.astype('float32')
signalStd = np.std(originalSignalNp)
correctedNoiseLevel = signalStd * 0.5

originalSignalTf = tf.convert_to_tensor(originalSignalNp)
originalSignalTf = originalSignalTf[tf.newaxis, :, tf.newaxis]

tf.random.set_seed(42)
signalWithNoise = addNoise(originalSignalTf, noiseLevel=correctedNoiseLevel)
signalScaled = scaleAmplitude(originalSignalTf, minScale=1.15, maxScale=1.2)
signalMasked = spectralMask(originalSignalTf, maskRatio=0.1)

# plot
plt.style.use('seaborn-v0_8-paper')
fig, axes = plt.subplots(4, 1, figsize=(14, 15), sharex=True)
fig.suptitle('Visualization of Data Augmentation Techniques', fontsize=20, fontweight='bold', y=0.98)

timeSteps = np.arange(originalSignalNp.shape[0])

axes[0].plot(timeSteps, originalSignalNp, label='Original Signal', color='black')
axes[0].set_title('A) Original Signal', fontsize=14, loc='left')
axes[0].set_ylabel('Amplitude (V)', fontsize=12)
axes[0].tick_params(axis='both', which='major', labelsize=10)

axes[1].plot(timeSteps, originalSignalNp, label='Original Signal', color='black', linestyle=':', alpha=0.8)
axes[1].plot(timeSteps, signalWithNoise[0, :, 0].numpy(), label='Augmented with Noise', color='#004c6d')
axes[1].set_title('B) Augmentation: Additive Noise', fontsize=14, loc='left')
axes[1].set_ylabel('Amplitude (V)', fontsize=12)
axes[1].tick_params(axis='both', which='major', labelsize=10)

axes[2].plot(timeSteps, originalSignalNp, label='Original Signal', color='black', linestyle=':', alpha=0.8)
axes[2].plot(timeSteps, signalScaled[0, :, 0].numpy(), label='Augmented by Scaling', color='#588622')
axes[2].set_title('C) Augmentation: Amplitude Scaling', fontsize=14, loc='left')
axes[2].set_ylabel('Amplitude (V)', fontsize=12)
axes[2].tick_params(axis='both', which='major', labelsize=10)

axes[3].plot(timeSteps, originalSignalNp, label='Original Signal', color='black', linestyle=':', alpha=0.8)
axes[3].plot(timeSteps, signalMasked[0, :, 0].numpy(), label='Augmented with Mask', color='#a50026')
axes[3].set_title('D) Augmentation: Spectral Masking', fontsize=14, loc='left')
axes[3].set_xlabel('Time Step in Window', fontsize=12)
axes[3].set_ylabel('Amplitude (V)', fontsize=12)
axes[3].tick_params(axis='both', which='major', labelsize=10)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('augmentation_visualization.png', dpi=300)
print("\nPlots saved as 'augmentation_visualization.png'")
plt.show()